# 🫀 Deteksi Dini Gangguan Tidur & Risiko Kardiovaskular
## Proyek EAS — Mata Kuliah Pembelajaran Mesin

| Info | Detail |
|------|--------|
| **Nama** | [Nama Mahasiswa] |
| **NRP** | [NRP Mahasiswa] |
| **Prodi** | [Nama Program Studi] |
| **Institusi** | Institut Teknologi Sepuluh Nopember (ITS) |

---

### 🎯 Latar Belakang
Gangguan tidur seperti **Insomnia** dan **Sleep Apnea** bukan hanya menurunkan kualitas hidup, tetapi juga menjadi faktor risiko signifikan penyakit kardiovaskular. Data dari perangkat *wearable* yang merekam detak jantung, tekanan darah, aktivitas fisik, dan pola tidur memungkinkan pendeteksian dini risiko tersebut melalui pendekatan *Machine Learning*.

### 📊 Dataset: Sleep Health and Lifestyle
Dataset publik dengan **400 observasi** dan **13 variabel** yang mencakup:
- Demografi: usia, jenis kelamin, pekerjaan
- Metrik Tidur: durasi, kualitas, jenis gangguan
- Metrik Kardiovaskular: detak jantung, tekanan darah, BMI
- Gaya Hidup: stres, aktivitas fisik, langkah harian

### 📌 Peta Alur Proyek

```
Dataset → EDA → Preprocessing → Feature Engineering → Modeling → Evaluasi → Streamlit Dashboard
```

| # | Tahap | Keterangan |
|---|-------|------------|
| 1 | **EDA** | Distribusi, korelasi, pola data |
| 2 | **Preprocessing** | Parsing, encoding, penanganan NaN |
| 3 | **Feature Engineering** | Skor Kardiovaskular, Indeks Kualitas Tidur |
| 4 | **Modeling** | Logistic Regression vs Random Forest |
| 5 | **Evaluasi** | Accuracy, F1, Confusion Matrix, Feature Importance |
| 6 | **Dashboard** | Aplikasi Streamlit interaktif |


In [ ]:
# ============================================================
# 📦 CELL 1: Instalasi Library
# ============================================================
!pip install pandas numpy matplotlib seaborn scikit-learn plotly joblib imbalanced-learn -q

print("✅ Semua library berhasil diinstall!")

In [ ]:
# ============================================================
# 📚 CELL 2: Import Library
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, accuracy_score,
                              f1_score, precision_score, recall_score)
import joblib, os, warnings
warnings.filterwarnings('ignore')

# Style settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi']     = 100
plt.rcParams['font.size']      = 11
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

import sklearn
print(f"✅ Library berhasil diimport!")
print(f"   pandas       : {pd.__version__}")
print(f"   numpy        : {np.__version__}")
print(f"   scikit-learn : {sklearn.__version__}")

In [ ]:
# ============================================================
# 📂 CELL 3: Load Dataset
# ============================================================
try:
    from google.colab import files
    print("📂 Silakan upload file 'sleep_health_lifestyle_dataset.csv'")
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])
except Exception:
    df = pd.read_csv('sleep_health_lifestyle_dataset.csv')

print(f"\n✅ Dataset berhasil dimuat!")
print(f"   Shape  : {df.shape[0]} baris × {df.shape[1]} kolom")
print(f"   Kolom  : {df.columns.tolist()}")
df.head(10)

In [ ]:
# ============================================================
# 👀 CELL 4: Eksplorasi Awal Dataset
# ============================================================
print("=" * 65)
print("  INFORMASI DATASET")
print("=" * 65)

print("\n📊 Tipe Data & Non-Null Count:")
df.info()

print("\n📈 Statistik Deskriptif Variabel Numerik:")
display(df.describe().round(2))

print("\n❓ Nilai Hilang (Missing Values):")
missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
mv_df = pd.DataFrame({'Jumlah': missing, 'Persentase (%)': missing_pct})
display(mv_df[mv_df['Jumlah'] > 0])

print("\n⚠️  Catatan: NaN pada 'Sleep Disorder' artinya TIDAK memiliki gangguan (Normal)")
print(f"\n🎯 Distribusi Target (Sleep Disorder):")
print(df['Sleep Disorder'].value_counts(dropna=False).to_string())

---
## 🔍 Bagian 2: Exploratory Data Analysis (EDA)

EDA bertujuan memahami distribusi data, hubungan antar variabel, dan pola yang relevan untuk pemodelan.

In [ ]:
# ============================================================
# 📊 CELL 5 — EDA: Distribusi Variabel Numerik
# ============================================================
numeric_cols = [
    'Age', 'Sleep Duration (hours)', 'Quality of Sleep (scale: 1-10)',
    'Physical Activity Level (minutes/day)', 'Stress Level (scale: 1-10)',
    'Heart Rate (bpm)', 'Daily Steps'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
axes = axes.flatten()
colors = sns.color_palette("husl", len(numeric_cols))

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=25, color=colors[i], edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean(),   color='red',  linestyle='--', lw=2,
                    label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='navy', linestyle=':',  lw=2,
                    label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(col.replace(' (', '\n('), fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Variabel Numerik', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=120, bbox_inches='tight')
plt.show()

# --- Insight ---
print("\n💡 Ringkasan Statistik Kunci:")
for col in ['Heart Rate (bpm)', 'Sleep Duration (hours)', 'Stress Level (scale: 1-10)']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    print(f"   {col[:30]:30s}  Mean={df[col].mean():.1f}  IQR=[{q1:.1f}–{q3:.1f}]  Skew={df[col].skew():.2f}")

In [ ]:
# ============================================================
# 📊 CELL 6 — EDA: Variabel Kategorikal & Target
# ============================================================
df_viz = df.copy()
df_viz['Sleep Disorder'] = df_viz['Sleep Disorder'].fillna('Normal')

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# --- 1. Target Pie Chart ---
slp_counts = df_viz['Sleep Disorder'].value_counts()
colors_pie  = ['#2ecc71', '#e74c3c', '#f39c12']
axes[0,0].pie(slp_counts.values, labels=slp_counts.index, autopct='%1.1f%%',
              colors=colors_pie[:len(slp_counts)], startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':11})
axes[0,0].set_title('Distribusi Sleep Disorder\n(Target Variable)', fontweight='bold')

# --- 2. Gender ---
gender_order = df_viz['Gender'].value_counts().index
sns.countplot(data=df_viz, x='Gender', order=gender_order,
              palette=['#3498db','#e91e8b'], ax=axes[0,1])
axes[0,1].bar_label(axes[0,1].containers[0], fontsize=11)
axes[0,1].set_title('Distribusi Jenis Kelamin', fontweight='bold')
axes[0,1].set_xlabel('')

# --- 3. BMI ---
bmi_order = ['Normal','Underweight','Overweight','Obese']
bmi_pal   = ['#2ecc71','#3498db','#f39c12','#e74c3c']
sns.countplot(data=df_viz, x='BMI Category', order=bmi_order,
              palette=bmi_pal, ax=axes[0,2])
axes[0,2].bar_label(axes[0,2].containers[0], fontsize=11)
axes[0,2].set_title('Distribusi Kategori BMI', fontweight='bold')
axes[0,2].set_xlabel('')
axes[0,2].tick_params(axis='x', rotation=10)

# --- 4. Occupation ---
occ_counts = df_viz['Occupation'].value_counts()
sns.barplot(x=occ_counts.values, y=occ_counts.index,
            ax=axes[1,0], palette='husl', orient='h')
for i, v in enumerate(occ_counts.values):
    axes[1,0].text(v+0.5, i, str(v), va='center', fontsize=10)
axes[1,0].set_title('Distribusi Pekerjaan', fontweight='bold')

# --- 5. Disorder by Gender ---
disorder_pal = {'Normal':'#2ecc71','Insomnia':'#e74c3c','Sleep Apnea':'#f39c12'}
sns.countplot(data=df_viz, x='Gender', hue='Sleep Disorder',
              palette=disorder_pal, ax=axes[1,1])
axes[1,1].set_title('Sleep Disorder × Gender', fontweight='bold')
axes[1,1].legend(title='Disorder', fontsize=9)
axes[1,1].set_xlabel('')

# --- 6. Disorder by BMI ---
sns.countplot(data=df_viz, x='BMI Category', hue='Sleep Disorder',
              order=bmi_order, palette=disorder_pal, ax=axes[1,2])
axes[1,2].set_title('Sleep Disorder × BMI', fontweight='bold')
axes[1,2].legend(title='Disorder', fontsize=9)
axes[1,2].set_xlabel('')
axes[1,2].tick_params(axis='x', rotation=10)

plt.suptitle('Analisis Variabel Kategorikal & Target', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 📊 CELL 7 — EDA: Box Plot (Numerik vs. Sleep Disorder)
# ============================================================
df_box   = df.copy()
df_box['Sleep Disorder'] = df_box['Sleep Disorder'].fillna('Normal')
key_vars = [
    'Heart Rate (bpm)', 'Sleep Duration (hours)',
    'Quality of Sleep (scale: 1-10)', 'Stress Level (scale: 1-10)',
    'Physical Activity Level (minutes/day)', 'Daily Steps'
]
disorder_order = ['Normal','Insomnia','Sleep Apnea']
d_palette      = {'Normal':'#2ecc71','Insomnia':'#e74c3c','Sleep Apnea':'#f39c12'}

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    sns.boxplot(data=df_box, x='Sleep Disorder', y=var, ax=axes[i],
                order=disorder_order, palette=d_palette, width=0.5,
                flierprops={'marker':'o','alpha':0.4,'ms':4})
    axes[i].set_title(var.split(' (')[0] + '\nberdasarkan Sleep Disorder',
                      fontweight='bold', fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Box Plot: Variabel Numerik × Gangguan Tidur', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n💡 Rata-rata per Kelompok:")
for var in ['Heart Rate (bpm)', 'Sleep Duration (hours)', 'Stress Level (scale: 1-10)']:
    g = df_box.groupby('Sleep Disorder')[var].mean().round(2)
    print(f"\n   {var}:")
    for k,v in g.items(): print(f"      {k:12s} → {v:.2f}")

In [ ]:
# ============================================================
# 📊 CELL 8 — EDA: Correlation Heatmap
# ============================================================
df_corr = df.copy()
df_corr['Sleep Disorder']  = df_corr['Sleep Disorder'].fillna('Normal')
df_corr['Disorder_Enc']    = df_corr['Sleep Disorder'].map({'Normal':0,'Insomnia':1,'Sleep Apnea':2})
df_corr['Gender_Enc']      = df_corr['Gender'].map({'Male':1,'Female':0})
df_corr[['Systolic','Diastolic']] = (
    df_corr['Blood Pressure (systolic/diastolic)'].str.split('/',expand=True).astype(int)
)

corr_cols = ['Gender_Enc','Age','Sleep Duration (hours)','Quality of Sleep (scale: 1-10)',
             'Physical Activity Level (minutes/day)','Stress Level (scale: 1-10)',
             'Systolic','Diastolic','Heart Rate (bpm)','Daily Steps','Disorder_Enc']

rename = {
    'Gender_Enc':'Gender','Sleep Duration (hours)':'Sleep Duration',
    'Quality of Sleep (scale: 1-10)':'Sleep Quality',
    'Physical Activity Level (minutes/day)':'Physical Activity',
    'Stress Level (scale: 1-10)':'Stress Level','Disorder_Enc':'Sleep Disorder'
}

corr_mat  = df_corr[corr_cols].corr()
corr_disp = corr_mat.rename(columns=rename, index=rename)
mask      = np.triu(np.ones_like(corr_disp, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_disp, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink":0.75},
            ax=ax, annot_kws={"size":9})
ax.set_title('Correlation Heatmap — Semua Variabel Numerik',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('eda_corr.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n🔑 Korelasi Tertinggi dengan Sleep Disorder (|r|):")
target_corr = corr_mat['Disorder_Enc'].drop('Disorder_Enc').abs().sort_values(ascending=False)
for col, val in target_corr.head(8).items():
    bar = '█' * int(val * 30)
    print(f"   {rename.get(col,col):35s} {bar}  r={val:.3f}")

---
## 🛠️ Bagian 3: Preprocessing & Feature Engineering

In [ ]:
# ============================================================
# 🛠️ CELL 9: Data Preprocessing
# ============================================================
df_proc = df.copy()

# --- 1. Handle Missing Values pada Target ---
print("1️⃣  Handling Missing Values...")
print(f"   Sebelum : {df_proc['Sleep Disorder'].isna().sum()} NaN")
df_proc['Sleep Disorder'] = df_proc['Sleep Disorder'].fillna('None')
print(f"   Sesudah : {df_proc['Sleep Disorder'].isna().sum()} NaN")
print(f"   Distribusi : {df_proc['Sleep Disorder'].value_counts().to_dict()}")

# --- 2. Parse Blood Pressure ---
print("\n2️⃣  Parsing Blood Pressure (systolic/diastolic)...")
df_proc[['Systolic','Diastolic']] = (
    df_proc['Blood Pressure (systolic/diastolic)']
    .str.split('/', expand=True).astype(int)
)
print(f"   Systolic  — Range: [{df_proc['Systolic'].min()}–{df_proc['Systolic'].max()}]  Mean: {df_proc['Systolic'].mean():.1f}")
print(f"   Diastolic — Range: [{df_proc['Diastolic'].min()}–{df_proc['Diastolic'].max()}]  Mean: {df_proc['Diastolic'].mean():.1f}")

# --- 3. Encode Categorical ---
print("\n3️⃣  Encoding Variabel Kategorikal...")
GENDER_MAP = {'Male':1, 'Female':0}
BMI_MAP    = {'Normal':0, 'Underweight':1, 'Overweight':2, 'Obese':3}
OCC_MAP    = {'Student':0, 'Office Worker':1, 'Manual Labor':2, 'Retired':3}

df_proc['Gender']       = df_proc['Gender'].map(GENDER_MAP)
df_proc['BMI Category'] = df_proc['BMI Category'].map(BMI_MAP)
df_proc['Occupation']   = df_proc['Occupation'].map(OCC_MAP)
print(f"   Gender       : {GENDER_MAP}")
print(f"   BMI Category : {BMI_MAP}")
print(f"   Occupation   : {OCC_MAP}")

# --- 4. Encode Target ---
print("\n4️⃣  Encoding Target Variable...")
TARGET_MAP = {'None':0, 'Insomnia':1, 'Sleep Apnea':2}
df_proc['Target'] = df_proc['Sleep Disorder'].map(TARGET_MAP)
print(f"   Mapping : {TARGET_MAP}")

# --- 5. Drop Unnecessary ---
df_proc.drop(columns=['Person ID','Blood Pressure (systolic/diastolic)','Sleep Disorder'],
             inplace=True)

print(f"\n✅ Preprocessing selesai.")
print(f"   Final shape : {df_proc.shape}")
print(f"   Distribusi target : {dict(df_proc['Target'].value_counts())}")

In [ ]:
# ============================================================
# ⚙️ CELL 10: Feature Engineering
# ============================================================
print("⚙️  Membuat Fitur Baru (Rekayasa Fitur)...\n")

# --- 1. Cardiovascular Risk Score (0–100) ---
# Kombinasi detak jantung + tekanan darah + stres
df_proc['Cardio_Risk_Score'] = (
    np.clip((df_proc['Heart Rate (bpm)']            - 50) / 50 * 40,  0, 40) +
    np.clip((df_proc['Systolic']                    - 90) / 90 * 35,  0, 35) +
    np.clip((df_proc['Diastolic']                   - 60) / 60 * 15,  0, 15) +
    np.clip(df_proc['Stress Level (scale: 1-10)']        / 10 * 10,  0, 10)
).clip(0, 100)
print(f"   ✔ Cardio_Risk_Score     : [{df_proc['Cardio_Risk_Score'].min():.1f} – {df_proc['Cardio_Risk_Score'].max():.1f}]  Mean={df_proc['Cardio_Risk_Score'].mean():.1f}")

# --- 2. Sleep Deprivation Index (0–100) ---
# Kekurangan durasi + rendahnya kualitas tidur
df_proc['Sleep_Deprivation_Index'] = (
    np.clip((8 - df_proc['Sleep Duration (hours)'].clip(0,8)) / 8 * 50, 0, 50) +
    np.clip((10 - df_proc['Quality of Sleep (scale: 1-10)'])  / 9 * 50, 0, 50)
).clip(0, 100)
print(f"   ✔ Sleep_Deprivation_Index: [{df_proc['Sleep_Deprivation_Index'].min():.1f} – {df_proc['Sleep_Deprivation_Index'].max():.1f}]  Mean={df_proc['Sleep_Deprivation_Index'].mean():.1f}")

# --- 3. Activity Score (0–100) ---
df_proc['Activity_Score'] = (
    np.clip(df_proc['Physical Activity Level (minutes/day)'] / 120 * 50, 0, 50) +
    np.clip(df_proc['Daily Steps']                          / 20000 * 50, 0, 50)
).clip(0, 100)
print(f"   ✔ Activity_Score         : [{df_proc['Activity_Score'].min():.1f} – {df_proc['Activity_Score'].max():.1f}]  Mean={df_proc['Activity_Score'].mean():.1f}")

print(f"\n   Total fitur : {df_proc.shape[1]-1} fitur  +  1 target")

# Visualisasi fitur baru
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
new_feats = ['Cardio_Risk_Score','Sleep_Deprivation_Index','Activity_Score']
target_labels = {0:'Normal', 1:'Insomnia', 2:'Sleep Apnea'}
pal = {0:'#2ecc71', 1:'#e74c3c', 2:'#f39c12'}

for i, feat in enumerate(new_feats):
    for tgt, lbl in target_labels.items():
        subset = df_proc[df_proc['Target']==tgt][feat]
        axes[i].hist(subset, bins=20, alpha=0.6, label=lbl, color=pal[tgt], edgecolor='white')
    axes[i].set_title(f'{feat}\n(per Kelas Disorder)', fontweight='bold', fontsize=10)
    axes[i].set_xlabel(feat.replace('_',' '))
    axes[i].legend(fontsize=9)

plt.suptitle('Distribusi Fitur Rekayasa × Kelas Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_engineering.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 🤖 Bagian 4: Pemodelan Machine Learning

Dua model klasifikasi dilatih dan dibandingkan: **Logistic Regression** dan **Random Forest**.

In [ ]:
# ============================================================
# ✂️ CELL 11: Train-Test Split & Feature Scaling
# ============================================================
FEATURE_COLS = [
    'Gender', 'Age', 'Occupation',
    'Sleep Duration (hours)', 'Quality of Sleep (scale: 1-10)',
    'Physical Activity Level (minutes/day)', 'Stress Level (scale: 1-10)',
    'BMI Category', 'Systolic', 'Diastolic', 'Heart Rate (bpm)',
    'Daily Steps', 'Cardio_Risk_Score', 'Sleep_Deprivation_Index', 'Activity_Score'
]

X = df_proc[FEATURE_COLS].values
y = df_proc['Target'].values

print("📦 Dataset Siap:")
print(f"   X shape  : {X.shape}")
print(f"   y shape  : {y.shape}")
print(f"   Kelas    : 0=Normal ({sum(y==0)}), 1=Insomnia ({sum(y==1)}), 2=Sleep Apnea ({sum(y==2)})")

# Stratified 80:20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n✂️  Train–Test Split (80:20, stratified):")
print(f"   Train : {X_train.shape[0]} sampel")
print(f"   Test  : {X_test.shape[0]} sampel")

# StandardScaler
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(X_train)
X_test_scaled   = scaler.transform(X_test)

print(f"\n⚖️  StandardScaler diterapkan (fit pada train, transform pada test)")
print(f"   Rata-rata 3 fitur pertama (train scaled): {X_train_scaled.mean(axis=0)[:3].round(3)}")
print(f"   Std 3 fitur pertama (train scaled)      : {X_train_scaled.std(axis=0)[:3].round(3)}")

In [ ]:
# ============================================================
# 📈 CELL 12: Model 1 — Logistic Regression
# ============================================================
print("=" * 60)
print("  MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)

lr_model = LogisticRegression(
    multi_class='multinomial', solver='lbfgs',
    max_iter=1000, C=1.0, random_state=42
)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average='weighted')
rec_lr = recall_score(y_test, y_pred_lr, average='weighted')
pre_lr = precision_score(y_test, y_pred_lr, average='weighted')
cv_lr  = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f"\n📊 Test Set Metrics:")
print(f"   Accuracy  : {acc_lr:.4f}  ({acc_lr*100:.2f}%)")
print(f"   F1-Score  : {f1_lr:.4f}")
print(f"   Precision : {pre_lr:.4f}")
print(f"   Recall    : {rec_lr:.4f}")
print(f"\n📉 Cross-Validation (5-Fold):")
print(f"   Scores  : {cv_lr.round(4)}")
print(f"   Mean    : {cv_lr.mean():.4f}")
print(f"   Std     : {cv_lr.std():.4f}")
print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Normal','Insomnia','Sleep Apnea']))

In [ ]:
# ============================================================
# 🌳 CELL 13: Model 2 — Random Forest Classifier
# ============================================================
print("=" * 60)
print("  MODEL 2: RANDOM FOREST CLASSIFIER")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf  = rf_model.predict_proba(X_test_scaled)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')
rec_rf = recall_score(y_test, y_pred_rf, average='weighted')
pre_rf = precision_score(y_test, y_pred_rf, average='weighted')
cv_rf  = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f"\n📊 Test Set Metrics:")
print(f"   Accuracy  : {acc_rf:.4f}  ({acc_rf*100:.2f}%)")
print(f"   F1-Score  : {f1_rf:.4f}")
print(f"   Precision : {pre_rf:.4f}")
print(f"   Recall    : {rec_rf:.4f}")
print(f"\n📉 Cross-Validation (5-Fold):")
print(f"   Scores  : {cv_rf.round(4)}")
print(f"   Mean    : {cv_rf.mean():.4f}")
print(f"   Std     : {cv_rf.std():.4f}")
print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal','Insomnia','Sleep Apnea']))

In [ ]:
# ============================================================
# ⚖️ CELL 14: Perbandingan Performa Model
# ============================================================
# --- Tabel Perbandingan ---
compare_df = pd.DataFrame({
    'Model'             : ['Logistic Regression', 'Random Forest'],
    'Test Accuracy'     : [acc_lr, acc_rf],
    'Weighted F1'       : [f1_lr,  f1_rf],
    'Weighted Precision': [pre_lr, pre_rf],
    'Weighted Recall'   : [rec_lr, rec_rf],
    'CV Mean (5-fold)'  : [cv_lr.mean(), cv_rf.mean()],
    'CV Std'            : [cv_lr.std(),  cv_rf.std()],
})
print("⚖️  Perbandingan Performa:")
display(compare_df.round(4).style.highlight_max(
    subset=['Test Accuracy','Weighted F1','CV Mean (5-fold)'],
    color='lightgreen', axis=0
))

# --- Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics_names = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
vals_lr = [acc_lr, f1_lr, pre_lr, rec_lr]
vals_rf = [acc_rf, f1_rf, pre_rf, rec_rf]
x = np.arange(len(metrics_names)); w = 0.35

b1 = axes[0].bar(x-w/2, vals_lr, w, label='Logistic Regression', color='#3498db', alpha=0.85)
b2 = axes[0].bar(x+w/2, vals_rf, w, label='Random Forest',       color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_names)
axes[0].set_ylim(0.5, 1.08); axes[0].set_ylabel('Score')
axes[0].set_title('Perbandingan Metrik Evaluasi', fontweight='bold')
axes[0].legend()
axes[0].bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
axes[0].bar_label(b2, fmt='%.3f', padding=3, fontsize=9)

# CV Box Plot
bp1 = axes[1].boxplot([cv_lr], positions=[1], widths=0.5, patch_artist=True,
                      boxprops=dict(facecolor='#3498db', alpha=0.7),
                      medianprops=dict(color='white', linewidth=2))
bp2 = axes[1].boxplot([cv_rf], positions=[2], widths=0.5, patch_artist=True,
                      boxprops=dict(facecolor='#e74c3c', alpha=0.7),
                      medianprops=dict(color='white', linewidth=2))
axes[1].set_xticks([1,2]); axes[1].set_xticklabels(['Logistic Regression','Random Forest'])
axes[1].set_title('Distribusi CV Accuracy (5-Fold)', fontweight='bold')
axes[1].set_ylabel('Accuracy')

plt.suptitle('Analisis Perbandingan Dua Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

best_model      = rf_model if acc_rf >= acc_lr else lr_model
best_model_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
best_acc        = max(acc_rf, acc_lr)
print(f"\n🏆 Model Terbaik: {best_model_name}  (Test Accuracy = {best_acc:.4f})")

---
## 📊 Bagian 5: Evaluasi Model Mendalam

In [ ]:
# ============================================================
# 📊 CELL 15: Confusion Matrix
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
class_names = ['Normal', 'Insomnia', 'Sleep Apnea']
pairs = [('Logistic Regression', y_pred_lr, 'Blues'),
         ('Random Forest',       y_pred_rf, 'Reds')]

for ax, (name, pred, cmap) in zip(axes, pairs):
    cm     = confusion_matrix(y_test, pred)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:,np.newaxis] * 100
    annot  = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)' for j in range(3)]
                        for i in range(3)])
    sns.heatmap(cm, annot=annot, fmt='', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='white',
                cbar_kws={"shrink":0.75})
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.suptitle('Confusion Matrix — Perbandingan Dua Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Metrics per kelas
print("\n📋 Metrik per Kelas — Random Forest:")
for i, cls in enumerate(class_names):
    mask = (y_test == i)
    tp = ((y_pred_rf==i) & mask).sum()
    fp = ((y_pred_rf==i) & ~mask).sum()
    fn = ((y_pred_rf!=i) & mask).sum()
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    print(f"   {cls:12s} — Precision:{prec:.3f}  Recall:{rec:.3f}  F1:{f1:.3f}  Support:{mask.sum()}")

In [ ]:
# ============================================================
# 📊 CELL 16: Feature Importance (Random Forest)
# ============================================================
feat_imp = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

colors = ['#e74c3c' if x>0.08 else '#f39c12' if x>0.04 else '#3498db'
          for x in feat_imp['Importance']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(feat_imp['Feature'], feat_imp['Importance'],
               color=colors, edgecolor='white', height=0.7)

for bar, val in zip(bars, feat_imp['Importance']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=9)

patches = [
    mpatches.Patch(color='#e74c3c', label='Tinggi  (> 8%)'),
    mpatches.Patch(color='#f39c12', label='Sedang  (4–8%)'),
    mpatches.Patch(color='#3498db', label='Rendah  (< 4%)')
]
ax.legend(handles=patches, loc='lower right', fontsize=10)
ax.set_xlabel('Importance Score (Mean Decrease Gini)', fontsize=12)
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlim(0, feat_imp['Importance'].max() * 1.25)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n🔑 Top 5 Fitur Paling Berpengaruh:")
top5 = feat_imp.sort_values('Importance', ascending=False).head(5)
for _, row in top5.iterrows():
    bar = '█' * int(row['Importance'] * 120)
    print(f"   {row['Feature']:40s} {bar} ({row['Importance']:.4f})")

---
## 💾 Bagian 6: Simpan Model & Artefak

In [ ]:
# ============================================================
# 💾 CELL 17: Simpan Model, Scaler, dan Metadata
# ============================================================
os.makedirs('models', exist_ok=True)

# Save model & scaler
joblib.dump(rf_model,  'models/model_rf.pkl')
joblib.dump(lr_model,  'models/model_lr.pkl')
joblib.dump(scaler,    'models/scaler.pkl')

# Save metadata (untuk digunakan di Streamlit)
metadata = {
    'features'      : FEATURE_COLS,
    'target_classes': {0:'Normal', 1:'Insomnia', 2:'Sleep Apnea'},
    'bmi_map'       : {'Normal':0, 'Underweight':1, 'Overweight':2, 'Obese':3},
    'occ_map'       : {'Student':0, 'Office Worker':1, 'Manual Labor':2, 'Retired':3},
    'gender_map'    : {'Male':1, 'Female':0},
    'rf_accuracy'   : round(acc_rf, 4),
    'lr_accuracy'   : round(acc_lr, 4),
}
joblib.dump(metadata, 'models/metadata.pkl')

print("💾 Artefak berhasil disimpan:")
print(f"   models/model_rf.pkl   — Random Forest (acc={acc_rf:.4f})")
print(f"   models/model_lr.pkl   — Logistic Regression (acc={acc_lr:.4f})")
print(f"   models/scaler.pkl     — StandardScaler")
print(f"   models/metadata.pkl   — Feature names & mappings")

# Download ke lokal (khusus Google Colab)
try:
    from google.colab import files
    print("\n📥 Mengunduh file model ke lokal...")
    for f in ['models/model_rf.pkl','models/scaler.pkl','models/metadata.pkl']:
        files.download(f)
    print("   ✅ Semua file berhasil diunduh! Simpan di folder 'models/' proyek Streamlit.")
except Exception:
    print("\n   ℹ️  Jalankan di Google Colab untuk auto-download.")

---
## 🌐 Bagian 7: Dashboard Streamlit

Jalankan aplikasi web interaktif dengan perintah:
```
streamlit run app.py
```

> **Catatan:** Pastikan file `model_rf.pkl`, `scaler.pkl`, dan `metadata.pkl` berada di dalam folder `models/` satu direktori dengan `app.py`.


In [ ]:
# ============================================================
# 🌐 CELL 18: Tulis Kode Streamlit ke app.py
# ============================================================
app_code = """import streamlit as st\nimport pandas as pd\nimport numpy as np\nimport plotly.graph_objects as go\nimport joblib, os\n\n# ── Konfigurasi ────────────────────────────────────────────────\nst.set_page_config(\n    page_title=\"Sleep & Cardio Risk Monitor\",\n    page_icon=\"🫀\", layout=\"wide\",\n    initial_sidebar_state=\"expanded\"\n)\nst.markdown(\"\"\"\n<style>\n.main-header{background:linear-gradient(135deg,#667eea,#764ba2);padding:1.5rem 2rem;\n border-radius:12px;color:white;text-align:center;margin-bottom:1.5rem}\n.rec-box{background:#f8f9fa;border-left:4px solid #667eea;padding:.9rem;\n border-radius:0 8px 8px 0;margin:.4rem 0;font-size:.9rem}\n</style>\n\"\"\", unsafe_allow_html=True)\n\n# ── Load Model ──────────────────────────────────────────────────\n@st.cache_resource\ndef load_resources():\n    base = \"models\"\n    model    = joblib.load(os.path.join(base,\"model_rf.pkl\"))\n    scaler   = joblib.load(os.path.join(base,\"scaler.pkl\"))\n    metadata = joblib.load(os.path.join(base,\"metadata.pkl\"))\n    return model, scaler, metadata\n\ntry:\n    model, scaler, meta = load_resources()\n    FEATURES = meta[\"features\"]\n    CLASSES  = meta[\"target_classes\"]\n    BMI_MAP  = meta[\"bmi_map\"]\n    OCC_MAP  = meta[\"occ_map\"]\n    GMAP     = meta[\"gender_map\"]\nexcept Exception as e:\n    st.error(f\"❌ Gagal memuat model: {e}\")\n    st.info(\"Pastikan folder 'models/' berisi model_rf.pkl, scaler.pkl, metadata.pkl\")\n    st.stop()\n\n# ── Header ──────────────────────────────────────────────────────\nst.markdown(\"\"\"\n<div class=\"main-header\">\n  <h1>🫀 Sleep &amp; Cardiovascular Risk Monitor</h1>\n  <p>Deteksi Dini Gangguan Tidur &amp; Risiko Kardiovaskular dari Data Wearable</p>\n  <p style=\"font-size:.85rem;opacity:.7\">Proyek EAS — Mata Kuliah Pembelajaran Mesin | ITS</p>\n</div>\"\"\", unsafe_allow_html=True)\n\n# ── Sidebar Input ───────────────────────────────────────────────\nwith st.sidebar:\n    st.header(\"📋 Masukkan Data Anda\")\n\n    st.subheader(\"👤 Informasi Pribadi\")\n    c1, c2 = st.columns(2)\n    age    = c1.slider(\"Usia\", 18, 90, 35)\n    gender = c2.selectbox(\"Kelamin\", [\"Pria\",\"Wanita\"])\n    occupation = st.selectbox(\"Pekerjaan\",\n                              [\"Student\",\"Office Worker\",\"Manual Labor\",\"Retired\"])\n    bmi_cat    = st.selectbox(\"Kategori BMI\",\n                              [\"Normal\",\"Underweight\",\"Overweight\",\"Obese\"])\n\n    st.divider()\n    st.subheader(\"💤 Data Tidur\")\n    sleep_dur  = st.slider(\"Durasi Tidur (jam)\",  4.0, 10.0, 7.0, 0.5)\n    sleep_qual = st.slider(\"Kualitas Tidur (1–10)\", 1, 10, 7)\n    stress     = st.slider(\"Tingkat Stres (1–10)\",  1, 10, 5)\n\n    st.divider()\n    st.subheader(\"❤️ Data Kardiovaskular\")\n    hr        = st.slider(\"Detak Jantung Istirahat (bpm)\", 50, 100, 70)\n    systolic  = st.slider(\"Tekanan Sistolik (mmHg)\", 90, 180, 120)\n    diastolic = st.slider(\"Tekanan Diastolik (mmHg)\", 60, 120, 80)\n\n    st.divider()\n    st.subheader(\"🏃 Aktivitas Fisik\")\n    phys_act = st.slider(\"Aktivitas Fisik (mnt/hari)\", 0, 120, 45)\n    steps    = st.slider(\"Langkah Harian\", 0, 20000, 8000, 500)\n\n# ── Feature Engineering ─────────────────────────────────────────\ngender_enc = GMAP.get(\"Male\" if gender==\"Pria\" else \"Female\", 0)\nocc_enc    = OCC_MAP.get(occupation, 0)\nbmi_enc    = BMI_MAP.get(bmi_cat, 0)\n\ncardio_risk = float(np.clip(\n    (hr - 50)/50*40 + (systolic-90)/90*35 +\n    (diastolic-60)/60*15 + stress/10*10, 0, 100))\n\nsleep_dep = float(np.clip(\n    (8-min(sleep_dur,8))/8*50 + (10-sleep_qual)/9*50, 0, 100))\n\nactivity_sc = float(np.clip(phys_act/120*50 + steps/20000*50, 0, 100))\n\nX_input = np.array([[\n    gender_enc, age, occ_enc, sleep_dur, sleep_qual, phys_act,\n    stress, bmi_enc, systolic, diastolic, hr, steps,\n    cardio_risk, sleep_dep, activity_sc\n]])\n\n# Predict\nX_scaled   = scaler.transform(X_input)\nprob       = model.predict_proba(X_scaled)[0]\npred_cls   = int(np.argmax(prob))\npred_label = CLASSES.get(pred_cls, \"Unknown\")\nrisk_score = float((1 - prob[0]) * 100)\n\n# ── Risk Category ───────────────────────────────────────────────\nif risk_score < 33:\n    risk_cat, risk_col = \"🟢 AMAN\",        \"#2ecc71\"\n    risk_desc = \"Risiko gangguan tidur & kardiovaskular Anda rendah. Pertahankan gaya hidup sehat!\"\nelif risk_score < 66:\n    risk_cat, risk_col = \"🟡 WASPADA\",     \"#f39c12\"\n    risk_desc = \"Terdapat indikasi risiko sedang. Pertimbangkan perbaikan pola tidur & stres.\"\nelse:\n    risk_cat, risk_col = \"🔴 RISIKO TINGGI\",\"#e74c3c\"\n    risk_desc = \"Risiko tinggi terdeteksi. Segera konsultasikan dengan profesional kesehatan.\"\n\n# ── Gauge + Hasil ───────────────────────────────────────────────\ncol_g, col_r = st.columns([1.3, 1])\n\nwith col_g:\n    gauge = go.Figure(go.Indicator(\n        mode=\"gauge+number\", value=round(risk_score,1),\n        domain={\"x\":[0,1],\"y\":[0,1]},\n        title={\"text\":\"Skor Risiko Kesehatan (%)\",\"font\":{\"size\":18}},\n        number={\"suffix\":\"%\",\"font\":{\"size\":44,\"color\":risk_col}},\n        gauge={\n            \"axis\":{\"range\":[0,100]},\n            \"bar\":{\"color\":risk_col,\"thickness\":0.3},\n            \"steps\":[\n                {\"range\":[0,33], \"color\":\"rgba(46,204,113,.15)\"},\n                {\"range\":[33,66],\"color\":\"rgba(243,156,18,.15)\"},\n                {\"range\":[66,100],\"color\":\"rgba(231,76,60,.15)\"}\n            ],\n            \"threshold\":{\"line\":{\"color\":risk_col,\"width\":5},\n                         \"thickness\":0.85,\"value\":risk_score}\n        }\n    ))\n    gauge.update_layout(height=320,\n                        margin=dict(l=30,r=30,t=50,b=20),\n                        paper_bgcolor=\"rgba(0,0,0,0)\")\n    st.plotly_chart(gauge, use_container_width=True)\n    st.markdown(\n        \"<div style='text-align:center'>\"\n        \"<span style='color:#2ecc71'>🟢 Aman (0–33)</span> &nbsp;|&nbsp; \"\n        \"<span style='color:#f39c12'>🟡 Waspada (33–66)</span> &nbsp;|&nbsp; \"\n        \"<span style='color:#e74c3c'>🔴 Tinggi (66–100)</span></div>\",\n        unsafe_allow_html=True\n    )\n\nwith col_r:\n    st.subheader(\"📊 Hasil Prediksi\")\n    st.markdown(\n        f\"<div style='background:rgba(0,0,0,.04);border:2px solid {risk_col};\"\n        f\"border-radius:10px;padding:1rem;text-align:center;margin-bottom:1rem'>\"\n        f\"<h3 style='color:{risk_col};margin:0'>{risk_cat}</h3>\"\n        f\"<p style='margin:.4rem 0 0;font-size:.9rem'>{risk_desc}</p></div>\",\n        unsafe_allow_html=True\n    )\n    fn = [st.success, st.warning, st.error][pred_cls]\n    fn(f\"{'✅' if pred_cls==0 else '⚠️' if pred_cls==1 else '❌'} Prediksi Kondisi: **{pred_label}**\")\n\n    st.subheader(\"📈 Distribusi Probabilitas\")\n    for lbl, p in zip([\"Normal\",\"Insomnia\",\"Sleep Apnea\"], prob):\n        cl, cb, cv = st.columns([2,4,1])\n        cl.write(lbl); cb.progress(float(p)); cv.write(f\"{p*100:.1f}%\")\n\n    st.subheader(\"🔬 Skor Derivasi\")\n    m1,m2,m3 = st.columns(3)\n    m1.metric(\"Cardio Risk\",       f\"{cardio_risk:.0f}/100\")\n    m2.metric(\"Sleep Deprivation\", f\"{sleep_dep:.0f}/100\")\n    m3.metric(\"Activity Score\",    f\"{activity_sc:.0f}/100\")\n\n# ── Rekomendasi ─────────────────────────────────────────────────\nst.divider()\nst.subheader(\"💡 Rekomendasi Personal\")\nrecs = []\nif sleep_dur  < 7:   recs.append((\"🌙\",\"Durasi Tidur\",\"Usahakan tidur **7–9 jam**/malam. Kurang tidur kronis meningkatkan risiko penyakit jantung.\"))\nif sleep_qual < 6:   recs.append((\"😴\",\"Kualitas Tidur\",\"Buat rutinitas tidur konsisten. Hindari layar & kafein 1–2 jam sebelum tidur.\"))\nif stress     > 7:   recs.append((\"🧘\",\"Manajemen Stres\",\"Coba meditasi, pernapasan dalam, atau yoga. Stres tinggi merusak siklus tidur.\"))\nif hr         > 85:  recs.append((\"❤️\",\"Detak Jantung\",\"RHR > 85 bpm perlu diperhatikan. Olahraga kardio rutin dapat menurunkan RHR secara signifikan.\"))\nif systolic   > 130 or diastolic > 85:\n    recs.append((\"💊\",\"Tekanan Darah\",\"Tekanan darah di atas normal. Kurangi garam & kafein, kelola stres. Konsultasikan ke dokter.\"))\nif phys_act   < 30:  recs.append((\"🏃\",\"Aktivitas Fisik\",\"WHO merekomendasikan **≥150 menit/minggu** aktivitas sedang. Mulai dari 30 menit/hari.\"))\nif steps      < 7000:recs.append((\"👣\",\"Langkah Kaki\",\"Targetkan **8.000–10.000 langkah/hari**. Jalan kaki adalah olahraga kardio yang mudah.\"))\nif bmi_cat in [\"Overweight\",\"Obese\"]:\n    recs.append((\"⚖️\",\"Berat Badan\",\"BMI ideal (18.5–24.9) penting untuk kesehatan jantung dan kualitas tidur.\"))\n\nif not recs:\n    st.balloons()\n    st.success(\"🌟 **Gaya hidup Anda sudah sangat baik!** Pertahankan pola hidup sehat ini.\")\nelse:\n    cols = st.columns(2)\n    for i,(ic,ttl,dsc) in enumerate(recs):\n        with cols[i%2]:\n            st.markdown(f'<div class=\"rec-box\"><strong>{ic} {ttl}</strong><br>{dsc}</div>',\n                        unsafe_allow_html=True)\n\n# ── Feature Importance ──────────────────────────────────────────\nst.divider()\nst.subheader(\"📊 Kontribusi Fitur (Random Forest)\")\nfi = pd.DataFrame({\"Fitur\":FEATURES,\"Importance\":model.feature_importances_}) \\\n       .sort_values(\"Importance\",ascending=True)\n\nfig_fi = go.Figure(go.Bar(\n    x=fi[\"Importance\"], y=fi[\"Fitur\"], orientation=\"h\",\n    marker=dict(color=fi[\"Importance\"],colorscale=\"RdYlGn\",\n                showscale=True,colorbar=dict(title=\"Score\"))\n))\nfig_fi.update_layout(title=\"Feature Importance — Random Forest\",\n                     xaxis_title=\"Importance Score\",height=450,\n                     margin=dict(l=10,r=10,t=50,b=10),\n                     paper_bgcolor=\"rgba(0,0,0,0)\")\nst.plotly_chart(fig_fi, use_container_width=True)\n\n# ── Footer ──────────────────────────────────────────────────────\nst.divider()\nfc1,fc2,fc3 = st.columns(3)\nfc1.metric(\"Model\",    \"Random Forest\")\nfc2.metric(\"Accuracy\", f\"{meta.get('rf_accuracy',0)*100:.2f}%\")\nfc3.metric(\"Fitur\",    str(len(FEATURES)))\nst.caption(\"⚠️ Aplikasi ini bersifat informatif dan TIDAK menggantikan diagnosis medis profesional.\")\nst.caption(\"📚 Proyek EAS — Mata Kuliah Pembelajaran Mesin | ITS\")\n"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py berhasil dibuat!")
print("\n📁 Struktur Proyek yang Diperlukan:")
print("   project/")
print("   ├── app.py")
print("   └── models/")
print("       ├── model_rf.pkl")
print("       ├── scaler.pkl")
print("       └── metadata.pkl")
print("\n🚀 Cara Menjalankan Streamlit:")
print("   1. Buka terminal di folder project/")
print("   2. pip install streamlit pandas plotly scikit-learn joblib")
print("   3. streamlit run app.py")
print("\n☁️  Atau jalankan di Streamlit Cloud:")
print("   1. Push semua file ke GitHub")
print("   2. Buka https://share.streamlit.io")
print("   3. Deploy repo Anda")


In [ ]:
# ============================================================
# ▶️ CELL 19 (Opsional): Jalankan Streamlit Langsung di Colab
# ============================================================
# Jalankan cell ini untuk preview Streamlit di Google Colab
# menggunakan LocalTunnel (tunnel publik sementara)

# !pip install streamlit pyngrok -q

# from pyngrok import ngrok
# !streamlit run app.py &
# import time; time.sleep(3)
# url = ngrok.connect(8501)
# print(f"\n🌐 Streamlit URL: {url}")

# ── Atau gunakan LocalTunnel ──
# !npx localtunnel --port 8501 &
# !streamlit run app.py

print("ℹ️  Uncomment baris di atas untuk menjalankan Streamlit langsung di Colab.")
print("   Disarankan menjalankan di lokal atau Streamlit Cloud untuk performa optimal.")